<a href="https://colab.research.google.com/github/DarkBlaze01-star/cs1281-labs/blob/main/2502110001(LAB_08).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IT12: Rough Cut - Parsing one log line
line = "  PE-26-0137 ; 25.02 ; DAY  "

# Extract fields
fields = line.strip().split(";")
code = fields[0].strip()
dia = float(fields[1])
shift = fields[2].strip().upper()

# Output results
print("Part Code:", code)
print("Diameter:", dia)
print("Shift:", shift)

# Proof that dia is a float
print("Proof (dia + 0.01):", dia + 0.01)

Part Code: PE-26-0137
Diameter: 25.02
Shift: DAY
Proof (dia + 0.01): 25.03


In [1]:
# 1. Define the 8 Hostile Test Inputs from Phase 1
hostile = [
    "",                             # T1: empty input
    "PE-26-0137 : 25.O2 : DAY",     # T2: letter 'O' in number
    "PE-26-0203 : 25.01",           # T3: missing field
    " PE-26-0201 : 24.97 : night ", # T4: wild whitespace
    "PE-XXXX : 24.97 : DAY",        # T5: unknown part code
    [],                             # T6: empty batch (not a string)
    "PE-26-0345 : 300.75 : DAY",    # T7: out-of-range value
    "   "                           # T8: blank line
]

rejects = []  # List to track bad lines and reasons
clean_records = []

# 2. Hardened Loop with Specific Exceptions
for line in hostile:
    # Check T1, T6, T8: Empty or invalid types
    if not isinstance(line, str) or not line.strip():
        rejects.append({"line": line, "reason": "Blank record or invalid type"})
        continue

    # Split fields
    fields = line.split(":")

    # Check T3: Missing fields
    if len(fields) < 3:
        rejects.append({"line": line, "reason": "Missing required fields"})
        continue

    # Clean whitespace (T4)
    code = fields[0].strip()
    dia_str = fields[1].strip()
    shift = fields[2].strip().upper()

    # Check T5: Unknown part code format
    if "XXXX" in code:
        rejects.append({"line": line, "reason": "Unknown part code"})
        continue

    # Check T2: Try converting string to float using NAMED EXCEPTION
    try:
        dia = float(dia_str)
    except ValueError:
        rejects.append({"line": line, "reason": f"Could not convert '{dia_str}' to float"})
        continue

    # Check T7: Out-of-range value guard
    if dia < 20.0 or dia > 30.0:
        rejects.append({"line": line, "reason": f"Out-of-range value ({dia})"})
        continue

    # If all checks pass, save the clean record
    clean_records.append({"code": code, "dia": dia, "shift": shift})

# 3. Print Results
print("=== CLEAN RECORDS ===")
for rec in clean_records:
    print(rec)

print("\n=== REJECTS TABLE ===")
for r in rejects:
    print(f"Line: {r['line']} | Reason: {r['reason']}")

=== CLEAN RECORDS ===
{'code': 'PE-26-0201', 'dia': 24.97, 'shift': 'NIGHT'}

=== REJECTS TABLE ===
Line:  | Reason: Blank record or invalid type
Line: PE-26-0137 : 25.O2 : DAY | Reason: Could not convert '25.O2' to float
Line: PE-26-0203 : 25.01 | Reason: Missing required fields
Line: PE-XXXX : 24.97 : DAY | Reason: Unknown part code
Line: [] | Reason: Blank record or invalid type
Line: PE-26-0345 : 300.75 : DAY | Reason: Out-of-range value (300.75)
Line:     | Reason: Blank record or invalid type


In [ ]:
# IT10: Standard - Processing 10 log lines
log_lines = [
    "  PE-26-0137 ; 25.02 ; DAY  ",
    " PE-26-0138 ; 24.98 ; DAY ",
    "PE-26-0139 ; 25.05 ; NIGHT",
    "  PE-26-0140 ; 24.92 ; DAY ",
    "PE-26-0141 ; 25.10 ; NIGHT ",
    " PE-26-0142 ; 25.00 ; DAY  ",
    "PE-26-0143 ; 24.95 ; DAY",
    "  PE-26-0144 ; 25.08 ; NIGHT ",
    "PE-26-0145 ; 24.90 ; DAY  ",
    " PE-26-0146 ; 25.03 ; NIGHT "
]

diameters = []

for line in log_lines:
    fields = line.strip().split(";")
    dia = float(fields[1])
    diameters.append(dia)


count = len(diameters)
avg_dia = sum(diameters) / count
min_dia = min(diameters)
max_dia = max(diameters)


print("=== IT10 SHIFT LOG REPORT ===")
print("Total Items Processed:", count)
print("Average Diameter:", round(avg_dia, 3))
print("Minimum Diameter:", min_dia)
print("Maximum Diameter:", max_dia)

=== IT10 SHIFT LOG REPORT ===
Total Items Processed: 10
Average Diameter: 25.003
Minimum Diameter: 24.9
Maximum Diameter: 25.1


In [2]:

class SpecViolation(Exception):
    """Custom exception raised when reading is out of acceptable spec range."""
    pass

class Machine:
    def __init__(self, name):
        self.name = name

    def process_reading(self, dia):

        if dia < 20.0 or dia > 30.0:
            raise SpecViolation(f"Reading {dia} is out of spec boundaries!")
        return dia

m = Machine("Lathe-01")
sample_readings = [24.97, 300.75]

for r in sample_readings:
    try:
        val = m.process_reading(r)
        print(f"Machine accepted reading: {val}")
    except SpecViolation as e:
        print(f"Machine rejected job: {e}")

Machine accepted reading: 24.97
Machine rejected job: Reading 300.75 is out of spec boundaries!


In [ ]:
# IT8: Fine - Binned Filtering & Shift Report
log_lines = [
    "  PE-26-0137 ; 25.02 ; DAY  ",
    " PE-26-0138 ; 24.98 ; DAY ",
    "PE-26-0139 ; 25.05 ; NIGHT",
    "  PE-26-0140 ; 24.92 ; DAY ",
    "PE-26-0141 ; 25.10 ; NIGHT ",
    " PE-26-0142 ; 25.00 ; DAY  ",
    "PE-26-0143 ; 24.95 ; DAY",
    "  PE-26-0144 ; 25.08 ; NIGHT ",
    "PE-26-0145 ; 24.90 ; DAY  ",
    " PE-26-0146 ; 25.03 ; NIGHT "
]

accepted = []
rework = []
scrap = []

scrapped_part_codes = []


for line in log_lines:
    fields = line.strip().split(";")
    code = fields[0].strip()
    dia = float(fields[1])

    if 24.98 <= dia <= 25.02:
        accepted.append(dia)
    elif (24.95 <= dia < 24.98) or (25.02 < dia <= 25.05):
        rework.append(dia)
    else:
        scrap.append(dia)
        scrapped_part_codes.append(code)

avg_accepted = sum(accepted) / len(accepted) if accepted else 0.0


print("=== IT8 QUALITY INSPECTION REPORT ===")
print("Accepted Count:", len(accepted))
print("Rework Count  :", len(rework))
print("Scrap Count   :", len(scrap))
print("Average Accepted Diameter:", round(avg_accepted, 3))
print("Scrapped Part Codes      :", scrapped_part_codes)

=== IT8 QUALITY INSPECTION REPORT ===
Accepted Count: 3
Rework Count  : 3
Scrap Count   : 4
Average Accepted Diameter: 25.0
Scrapped Part Codes      : ['PE-26-0140', 'PE-26-0141', 'PE-26-0144', 'PE-26-0145']


In [ ]:
# IT6: Precision - Shift Comparison using Sets
log_lines = [
    "  PE-26-0137 ; 25.02 ; DAY  ",
    " PE-26-0138 ; 24.98 ; DAY ",
    "PE-26-0139 ; 25.05 ; NIGHT",
    "  PE-26-0140 ; 24.92 ; DAY ",
    "PE-26-0141 ; 25.10 ; NIGHT ",
    " PE-26-0142 ; 25.00 ; DAY  ",
    "PE-26-0143 ; 24.95 ; DAY",
    "  PE-26-0144 ; 25.08 ; NIGHT ",
    "PE-26-0145 ; 24.90 ; DAY  ",
    " PE-26-0146 ; 25.03 ; NIGHT "
]

# Trackers for Day and Night shifts
day_total = 0
day_accepted = 0
night_total = 0
night_accepted = 0

# Sets to store unique scrapped part codes per shift
# Comment: Sets are used here because they enforce uniqueness and enable set operations (intersection/difference).
day_scrap_codes = set()
night_scrap_codes = set()

for line in log_lines:
    fields = line.strip().split(";")
    code = fields[0].strip()
    dia = float(fields[1])
    shift = fields[2].strip().upper()

    is_accepted = (24.98 <= dia <= 25.02)
    is_rework = (24.95 <= dia < 24.98) or (25.02 < dia <= 25.05)
    is_scrap = not (is_accepted or is_rework)

    if shift == "DAY":
        day_total += 1
        if is_accepted:
            day_accepted += 1
        elif is_scrap:
            day_scrap_codes.add(code)
    elif shift == "NIGHT":
        night_total += 1
        if is_accepted:
            night_accepted += 1
        elif is_scrap:
            night_scrap_codes.add(code)

# Calculate acceptance rates
day_acc_rate = (day_accepted / day_total * 100) if day_total else 0.0
night_acc_rate = (night_accepted / night_total * 100) if night_total else 0.0

# Set operations
scrapped_in_both = day_scrap_codes.intersection(night_scrap_codes)
day_only_scrap = day_scrap_codes.difference(night_scrap_codes)
night_only_scrap = night_scrap_codes.difference(day_scrap_codes)

# Output Report
print("=== IT6 SHIFT COMPARISON REPORT ===")
print(f"Day Shift Acceptance Rate  : {day_acc_rate:.1f}% ({day_accepted}/{day_total})")
print(f"Night Shift Acceptance Rate: {night_acc_rate:.1f}% ({night_accepted}/{night_total})")
print("Scrapped in BOTH shifts     :", scrapped_in_both)
print("Scrapped ONLY in Day shift  :", day_only_scrap)
print("Scrapped ONLY in Night shift:", night_only_scrap)

=== IT6 SHIFT COMPARISON REPORT ===
Day Shift Acceptance Rate  : 50.0% (3/6)
Night Shift Acceptance Rate: 0.0% (0/4)
Scrapped in BOTH shifts     : set()
Scrapped ONLY in Day shift  : {'PE-26-0145', 'PE-26-0140'}
Scrapped ONLY in Night shift: {'PE-26-0141', 'PE-26-0144'}
